# Part 1: Lead qualification decision model

Fine-tunes `LiquidAI/LFM3-1.7B` with Unsloth's Clef decision head on LeadForge's synthetic CRM data. Companion to the tutorial on raoofaltaher.com.

In [1]:
import os
import time
from pathlib import Path

if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)

HF_USERNAME = "raoofaltaher"
PUBLISH = True
LOAD_IN_4BIT = True
TIMINGS = {}

print("Working directory:", Path.cwd())

Working directory: /home/dev/work/lead-decision-models


## Environment

In [2]:
import platform
import torch

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA runtime:", torch.version.cuda)
else:
    raise RuntimeError("A supported GPU runtime is required for this training run.")

Python: 3.13.18
PyTorch: 2.14.1+cu132
CUDA available: True
GPU: NVIDIA GeForce RTX 5090 Laptop GPU
CUDA runtime: 13.2


## Data

In [3]:
import hashlib
import shutil
from pathlib import Path
from urllib.request import urlopen

from datasets import load_dataset

DATASET_COMMIT = "39ebe8c9e6cf25fe33a003662e18557b4de55142"
TASK_BASE = (
    "https://raw.githubusercontent.com/leadforge-dev/leadforge-datasets/"
    f"{DATASET_COMMIT}/releases/v0.1.0-alpha/intermediate/"
    "tasks/converted_within_90_days"
)

# SHA-256 values from the pinned intermediate manifest.
TASK_FILES = {
    "train": ("train.parquet", "ba7dd8f6ff24add8d6d4897d9870f46e818bb0ca3432a59c263c2d4e16f79184"),
    "validation": ("valid.parquet", "61029542d352f493b56d1ad85cf52a4d4970c651ed87cf5bdf05978d222e3592"),
    "test": ("test.parquet", "d428c07decc2b1fdf8b5f56a1a63c65799897f1e22b61afd9f5d517f74593f09"),
}

DATA_DIR = Path("data/leadforge-intermediate")
DATA_DIR.mkdir(parents=True, exist_ok=True)

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

local_files = {}

for split_name, (filename, expected_sha256) in TASK_FILES.items():
    destination = DATA_DIR / filename
    url = f"{TASK_BASE}/{filename}"

    with urlopen(url, timeout=60) as response, destination.open("wb") as output:
        shutil.copyfileobj(response, output)

    actual_sha256 = sha256_file(destination)
    if actual_sha256 != expected_sha256:
        raise ValueError(
            f"{split_name} checksum mismatch: expected {expected_sha256}, "
            f"received {actual_sha256}"
        )

    local_files[split_name] = str(destination)
    print(split_name, "SHA-256 verified:", actual_sha256)

lead_splits = load_dataset(
    "parquet",
    data_files=local_files,
)

for split_name, split in lead_splits.items():
    print(split_name, len(split), split.column_names)

train SHA-256 verified: ba7dd8f6ff24add8d6d4897d9870f46e818bb0ca3432a59c263c2d4e16f79184


validation SHA-256 verified: 61029542d352f493b56d1ad85cf52a4d4970c651ed87cf5bdf05978d222e3592


test SHA-256 verified: d428c07decc2b1fdf8b5f56a1a63c65799897f1e22b61afd9f5d517f74593f09


Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

train 3500 ['account_id', 'industry', 'region', 'employee_band', 'estimated_revenue_band', 'process_maturity_band', 'contact_id', 'role_function', 'seniority', 'buyer_role', 'lead_id', 'lead_created_at', 'lead_source', 'first_touch_channel', 'touch_count', 'inbound_touch_count', 'outbound_touch_count', 'session_count', 'pricing_page_views', 'demo_page_views', 'total_session_duration_seconds', 'touches_week_1', 'touches_last_7_days', 'days_since_first_touch', 'activity_count', 'days_since_last_touch', 'opportunity_created', 'has_open_opportunity', 'opportunity_estimated_acv', 'expected_acv', 'total_touches_all', 'converted_within_90_days']
validation 750 ['account_id', 'industry', 'region', 'employee_band', 'estimated_revenue_band', 'process_maturity_band', 'contact_id', 'role_function', 'seniority', 'buyer_role', 'lead_id', 'lead_created_at', 'lead_source', 'first_touch_channel', 'touch_count', 'inbound_touch_count', 'outbound_touch_count', 'session_count', 'pricing_page_views', 'demo_

In [4]:
import pandas as pd

LABEL_COLUMN = "converted_within_90_days"

frames = []
for split_name, split in lead_splits.items():
    frame = split.to_pandas()
    frame["release_split"] = split_name
    frames.append(frame)

leads = pd.concat(frames, ignore_index=True)

print("Rows:", len(leads))
print("Unique leads:", leads["lead_id"].nunique())
print("Target distribution:")
print(leads[LABEL_COLUMN].value_counts(dropna=False))
print("Target rate:")
print(leads[LABEL_COLUMN].mean())

assert len(leads) == 5000
assert leads["lead_id"].is_unique
assert leads[LABEL_COLUMN].notna().all()

Rows: 5000
Unique leads: 5000
Target distribution:
converted_within_90_days
False    3951
True     1049
Name: count, dtype: int64
Target rate:
0.2098


In [5]:
LEADS_TABLE_SHA256 = "7a4bf1b7afeb54e0b8d74b7f89b309ceee83455f734ca67f59e94e32d430fa95"
LEADS_TABLE_URL = (
    "https://raw.githubusercontent.com/leadforge-dev/leadforge-datasets/"
    f"{DATASET_COMMIT}/releases/v0.1.0-alpha/intermediate/tables/leads.parquet"
)

leads_table_path = DATA_DIR / "leads.parquet"

with urlopen(LEADS_TABLE_URL, timeout=60) as response, leads_table_path.open("wb") as output:
    shutil.copyfileobj(response, output)

if sha256_file(leads_table_path) != LEADS_TABLE_SHA256:
    raise ValueError("leads.parquet checksum mismatch")

conversion_times = pd.read_parquet(
    leads_table_path,
    columns=["lead_id", "conversion_timestamp"],
)

timed_leads = leads.merge(
    conversion_times,
    on="lead_id",
    how="left",
    validate="one_to_one",
)

assert timed_leads["conversion_timestamp"].notna().eq(
    timed_leads[LABEL_COLUMN].astype(bool)
).all()

created_at = pd.to_datetime(timed_leads["lead_created_at"])
converted_at = pd.to_datetime(timed_leads["conversion_timestamp"])
days_to_conversion = (converted_at - created_at).dt.days

converted_by_day30 = days_to_conversion.le(30)

clean_leads = timed_leads.loc[~converted_by_day30].copy()
clean_leads["converted_after_day30"] = days_to_conversion.loc[~converted_by_day30].lt(90)

print("Published positive labels:", int(timed_leads[LABEL_COLUMN].sum()))
print("Converted on or before day 30:", int(converted_by_day30.sum()))
print("Leads in the clean risk set:", len(clean_leads))
print("Clean positives:", int(clean_leads["converted_after_day30"].sum()))

Published positive labels: 1049
Converted on or before day 30: 22
Leads in the clean risk set: 4978
Clean positives: 1027


In [6]:
MODEL_FEATURES = [
    "industry",
    "region",
    "employee_band",
    "estimated_revenue_band",
    "process_maturity_band",
    "role_function",
    "seniority",
    "buyer_role",
    "touch_count",
    "inbound_touch_count",
    "outbound_touch_count",
    "session_count",
    "pricing_page_views",
    "demo_page_views",
    "total_session_duration_seconds",
    "touches_week_1",
    "touches_last_7_days",
    "days_since_first_touch",
    "activity_count",
    "days_since_last_touch",
    "opportunity_created",
    "has_open_opportunity",
    "opportunity_estimated_acv",
    "expected_acv",
]

required_columns = {
    *MODEL_FEATURES,
    "account_id",
    "contact_id",
    "lead_id",
    LABEL_COLUMN,
}

missing_columns = required_columns - set(leads.columns)
if missing_columns:
    raise ValueError(f"Dataset schema changed. Missing columns: {sorted(missing_columns)}")

assert "total_touches_all" in leads.columns
assert "total_touches_all" not in MODEL_FEATURES
assert LABEL_COLUMN not in MODEL_FEATURES
assert "account_id" not in MODEL_FEATURES
assert "contact_id" not in MODEL_FEATURES
assert "lead_id" not in MODEL_FEATURES

In [7]:
print(leads[MODEL_FEATURES].isna().sum().loc[lambda counts: counts > 0])

touch_count                        400
inbound_touch_count                416
outbound_touch_count               403
session_count                      373
pricing_page_views                 398
demo_page_views                    384
total_session_duration_seconds     400
touches_week_1                     411
touches_last_7_days                406
days_since_first_touch             465
activity_count                     389
days_since_last_touch              469
opportunity_estimated_acv         3049
expected_acv                       391
dtype: int64


## Grouped splits

In [8]:
from itertools import combinations
from sklearn.model_selection import StratifiedGroupKFold
import numpy as np

release_frames = {
    name: split.to_pandas()
    for name, split in lead_splits.items()
}

for left, right in combinations(release_frames, 2):
    left_accounts = set(release_frames[left]["account_id"].dropna())
    right_accounts = set(release_frames[right]["account_id"].dropna())
    left_contacts = set(release_frames[left]["contact_id"].dropna())
    right_contacts = set(release_frames[right]["contact_id"].dropna())

    print(
        left,
        right,
        "shared accounts:",
        len(left_accounts & right_accounts),
        "shared contacts:",
        len(left_contacts & right_contacts),
    )

if leads["account_id"].isna().any():
    raise ValueError("Account IDs are required for the grouped evaluation split.")

splitter = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=3407,
)

folds = list(
    splitter.split(
        leads,
        y=leads[LABEL_COLUMN].astype(int),
        groups=leads["account_id"],
    )
)

test_indices = folds[0][1]
validation_indices = folds[1][1]
train_indices = np.concatenate([folds[2][1], folds[3][1], folds[4][1]])

train_df = leads.iloc[train_indices].reset_index(drop=True)
validation_df = leads.iloc[validation_indices].reset_index(drop=True)
test_df = leads.iloc[test_indices].reset_index(drop=True)

def group_ids(frame, column):
    return set(frame[column].dropna())

for column in ["account_id", "contact_id"]:
    assert group_ids(train_df, column).isdisjoint(group_ids(validation_df, column))
    assert group_ids(train_df, column).isdisjoint(group_ids(test_df, column))
    assert group_ids(validation_df, column).isdisjoint(group_ids(test_df, column))

for name, frame in [
    ("train", train_df),
    ("validation", validation_df),
    ("test", test_df),
]:
    print(
        name,
        "rows:", len(frame),
        "accounts:", frame["account_id"].nunique(),
        "conversion rate:", frame[LABEL_COLUMN].mean(),
    )

train validation shared accounts: 487 shared contacts: 372
train test shared accounts: 518 shared contacts: 403
validation test shared accounts: 243 shared contacts: 110
train rows: 2998 accounts: 767 conversion rate: 0.209472981987992
validation rows: 1000 accounts: 262 conversion rate: 0.21
test rows: 1002 accounts: 251 conversion rate: 0.21057884231536927


In [9]:
calibration_splitter = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=3408,
)

fit_indices, calibration_indices = next(
    calibration_splitter.split(
        train_df,
        y=train_df[LABEL_COLUMN].astype(int),
        groups=train_df["account_id"],
    )
)

lead_fit_df = train_df.iloc[fit_indices].reset_index(drop=True)
lead_calibration_df = train_df.iloc[calibration_indices].reset_index(drop=True)

assert group_ids(lead_fit_df, "account_id").isdisjoint(
    group_ids(lead_calibration_df, "account_id")
)
assert group_ids(lead_fit_df, "contact_id").isdisjoint(
    group_ids(lead_calibration_df, "contact_id")
)

for name, frame in [
    ("optimizer training", lead_fit_df),
    ("calibration", lead_calibration_df),
]:
    if frame[LABEL_COLUMN].nunique() != 2:
        raise ValueError(
            f"{name} split must contain both conversion classes. "
            "Collect more independent accounts or revise the grouped split."
        )

    print(name, "leads:", len(frame))
    print(name, "accounts:", frame["account_id"].nunique())
    print(name, "conversion rate:", frame[LABEL_COLUMN].mean())

optimizer training leads: 2400
optimizer training accounts: 614
optimizer training conversion rate: 0.20958333333333334
calibration leads: 598
calibration accounts: 153
calibration conversion rate: 0.20903010033444816


## ICP policy and decision records

In [10]:
TARGET_INDUSTRIES = {
    "manufacturing",
    "logistics",
    "professional_services",
}

TARGET_REGIONS = {"US", "UK"}

TARGET_EMPLOYEE_BANDS = {
    "200-499",
    "500-999",
    "1000-1999",
}

def icp_label(row):
    required = ["industry", "region", "employee_band"]

    allowed_values = {
        "industry": TARGET_INDUSTRIES,
        "region": TARGET_REGIONS,
        "employee_band": TARGET_EMPLOYEE_BANDS,
    }

    known_mismatch = any(
        not pd.isna(row.get(column))
        and row[column] not in allowed
        for column, allowed in allowed_values.items()
    )

    if known_mismatch:
        return "not_fit"

    if any(pd.isna(row.get(column)) for column in required):
        return "insufficient_evidence"

    return "fit"

ICP_CLASSES = ["fit", "not_fit", "insufficient_evidence"]
TRAINABLE_ICP_CLASSES = ["fit", "not_fit"]

for name, frame in [
    ("optimizer training", lead_fit_df),
    ("calibration", lead_calibration_df),
    ("validation", validation_df),
    ("test", test_df),
]:
    labels = frame.apply(icp_label, axis=1)
    missing = set(TRAINABLE_ICP_CLASSES) - set(labels.unique())
    if missing:
        raise ValueError(
            f"{name} split is missing ICP classes: {sorted(missing)}. "
            "Review the policy and grouped split sizes before training."
        )

    print(name, labels.value_counts().reindex(ICP_CLASSES, fill_value=0).to_dict())

optimizer training {'fit': 1610, 'not_fit': 790, 'insufficient_evidence': 0}
calibration {'fit': 383, 'not_fit': 215, 'insufficient_evidence': 0}
validation {'fit': 683, 'not_fit': 317, 'insufficient_evidence': 0}
test {'fit': 672, 'not_fit': 330, 'insufficient_evidence': 0}


In [11]:
LEAD_QUESTIONS = {
    "icp_fit": {
        "type": "choice",
        "instructions": (
            "Apply the stated fictional ICP using only industry, region and employee_band. "
            "Ignore contact role, seniority, lead source, engagement and opportunity fields "
            "when deciding account fit. Return not_fit if any known required value is "
            "outside the target. Otherwise use insufficient_evidence when a required "
            "value is missing. Do not infer facts that are not present."
        ),
        "criteria": {
            "fit": "Target industry, US or UK region, and an employee band from 200 to 1999.",
            "not_fit": "At least one known firmographic value falls outside the target ICP.",
            "insufficient_evidence": "At least one required field is missing, and no known value rules out ICP fit.",
        },
    },
    "conversion_outcome_90d": {
        "type": "noul",
        "instructions": (
            "Classify whether the simulator records a closed_won event within 90 days "
            "of lead creation. Features cover only the first 30 days, so the label "
            "may include outcomes that occurred before the feature cutoff. "
            "Do not interpret this as a day-30-to-day-90 forecast."
        ),
    },
}

In [12]:
import json
from datasets import Dataset

def to_python_value(value):
    if pd.isna(value):
        return None
    if hasattr(value, "item"):
        return value.item()
    return value

def make_lead_record(row):
    feature_state = {
        name: to_python_value(row[name])
        for name in MODEL_FEATURES
    }

    state = {
        "task": "B2B lead qualification",
        "snapshot": "Features available through day 30 after lead creation.",
        "features": feature_state,
    }

    return {
        "state": json.dumps(state, sort_keys=True, ensure_ascii=False),
        "questions": LEAD_QUESTIONS,
        "gold": {
            "icp_fit": icp_label(row),
            "conversion_outcome_90d": bool(row[LABEL_COLUMN]),
        },
    }

def to_decision_dataset(frame, builder):
    rows = [
        builder(row)
        for row in frame.to_dict(orient="records")
    ]
    return Dataset.from_list(rows)

lead_train_raw = to_decision_dataset(lead_fit_df, make_lead_record)
lead_calibration_raw = to_decision_dataset(lead_calibration_df, make_lead_record)
lead_validation_raw = to_decision_dataset(validation_df, make_lead_record)
lead_test_raw = to_decision_dataset(test_df, make_lead_record)

print(lead_train_raw[0])

{'state': '{"features": {"activity_count": null, "buyer_role": "technical_evaluator", "days_since_first_touch": 30.333835742092244, "days_since_last_touch": 11.003359485085898, "demo_page_views": 0.0, "employee_band": "500-999", "estimated_revenue_band": "$10M-$50M", "expected_acv": 58295.50663158452, "has_open_opportunity": false, "inbound_touch_count": 5.0, "industry": "logistics", "opportunity_created": false, "opportunity_estimated_acv": null, "outbound_touch_count": 0.0, "pricing_page_views": 0.0, "process_maturity_band": "high", "region": "UK", "role_function": "it_director", "seniority": "c_suite", "session_count": 2.0, "total_session_duration_seconds": 632.0, "touch_count": 5.0, "touches_last_7_days": null, "touches_week_1": 1.0}, "snapshot": "Features available through day 30 after lead creation.", "task": "B2B lead qualification"}', 'questions': {'icp_fit': {'type': 'choice', 'instructions': 'Apply the stated fictional ICP using only industry, region and employee_band. Ignore

In [13]:
for name, dataset in [
    ("train", lead_train_raw),
    ("calibration", lead_calibration_raw),
    ("validation", lead_validation_raw),
    ("test", lead_test_raw),
]:
    for row in dataset:
        assert set(row["questions"]) == set(row["gold"])
        assert row["gold"]["icp_fit"] in {
            "fit",
            "not_fit",
            "insufficient_evidence",
        }
        assert isinstance(row["gold"]["conversion_outcome_90d"], bool)

    print(name, len(dataset))

train 2400
calibration 598
validation 1000
test 1002


## Model

In [14]:
from unsloth import (
    FastDecisionModel,
    DecisionTrainer,
    is_bfloat16_supported,
)
from transformers import TrainingArguments

MODEL_ID = "LiquidAI/LFM3-1.7B"
MODEL_REVISION = "ac2586a34f1a1c03329dbdcdefabd8b6478ddaa7"

print("Model:", MODEL_ID)
print("Pinned revision:", MODEL_REVISION)

model, tokenizer = FastDecisionModel.from_pretrained(
    model_name=MODEL_ID,
    revision=MODEL_REVISION,
    trust_remote_code=True,
    max_seq_length=2048,
    load_in_4bit=LOAD_IN_4BIT,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


W1103 15:07:28.636000 1051 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


W1103 15:07:28.668000 1051 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


/home/dev/work/lead-decision-models/unsloth_env/lib/python3.13/site-packages/transformers/configuration_utils.py:723: FutureWarning: `rope_scaling` is deprecated and will be removed in a future version. Set `rope_parameters` instead, with `rope_type` and `rope_theta` included.
  warnings.warn(


🦥 Unsloth Zoo will now patch everything to make training faster!


Model: LiquidAI/LFM3-1.7B
Pinned revision: ac2586a34f1a1c03329dbdcdefabd8b6478ddaa7


... (2 outputs left out)

Unsloth: Repaired remote modeling code so it trains: Lfm3ForCausalLM.forward.


Unsloth: compiling each transformer block as one region (Lfm3DecoderLayer).


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

In [15]:
model = FastDecisionModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0.0,
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

In [16]:
lead_train_items_all, lead_train_report = FastDecisionModel.build_dataset(
    lead_train_raw,
    tokenizer,
    model,
)

lead_calibration_items, lead_calibration_report = FastDecisionModel.build_dataset(
    lead_calibration_raw,
    tokenizer,
    model,
)

lead_validation_items, lead_validation_report = FastDecisionModel.build_dataset(
    lead_validation_raw,
    tokenizer,
    model,
)

lead_test_items, lead_test_report = FastDecisionModel.build_dataset(
    lead_test_raw,
    tokenizer,
    model,
)

for name, report in [
    ("train", lead_train_report),
    ("calibration", lead_calibration_report),
    ("validation", lead_validation_report),
    ("test", lead_test_report),
]:
    print(name, report)
    if report["skipped"]:
        raise ValueError(
            f"{name} dataset contains skipped decisions: {report['reason']}"
        )

train {'total': 4800, 'skipped': 0, 'reason': None, 'truncated': 0}
calibration {'total': 1196, 'skipped': 0, 'reason': None, 'truncated': 0}
validation {'total': 2000, 'skipped': 0, 'reason': None, 'truncated': 0}
test {'total': 2004, 'skipped': 0, 'reason': None, 'truncated': 0}


In [17]:
lead_train_items = lead_train_items_all

print("Training rows:", len(lead_train_items))
print("Calibration rows:", len(lead_calibration_items))
print("Validation rows:", len(lead_validation_items))
print("Test rows:", len(lead_test_items))

from collections import Counter

lead_calibration_counts = Counter(
    question_type
    for item in lead_calibration_items
    for question_type in item["qtypes"]
)

if (
    len(lead_calibration_counts) != len(LEAD_QUESTIONS)
    or min(lead_calibration_counts.values()) < 10
):
    raise ValueError(
        "Provide at least ten calibration decisions for both lead question types. "
        f"Observed counts: {dict(lead_calibration_counts)}"
    )

Training rows: 2400
Calibration rows: 598
Validation rows: 1000
Test rows: 1002


## Baselines

In [18]:
lead_baseline = FastDecisionModel.evaluate(
    model,
    tokenizer,
    lead_validation_items,
)

print("Untrained decision-head validation metrics:", lead_baseline)

Untrained decision-head validation metrics: {'accuracy': 0.395, 'ece': 0.31961952917277814, 'loss': 0.9056575514376164, 'record_accuracy': 0.0}


In [19]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score

CATEGORICAL_FEATURES = [
    "industry",
    "region",
    "employee_band",
    "estimated_revenue_band",
    "process_maturity_band",
    "role_function",
    "seniority",
    "buyer_role",
]

NUMERIC_FEATURES = [
    name
    for name in MODEL_FEATURES
    if name not in CATEGORICAL_FEATURES
]

numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="constant",
                fill_value="__missing__",
            ),
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                min_frequency=10,
            ),
        ),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, NUMERIC_FEATURES),
        ("categorical", categorical_pipeline, CATEGORICAL_FEATURES),
    ]
)

logistic_baseline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=2000)),
    ]
)

logistic_baseline.fit(
    lead_fit_df[MODEL_FEATURES],
    lead_fit_df[LABEL_COLUMN].astype(int),
)

positive_class_index = list(
    logistic_baseline.named_steps["classifier"].classes_
).index(1)

logistic_validation_probabilities = logistic_baseline.predict_proba(
    validation_df[MODEL_FEATURES]
)[:, positive_class_index]

validation_y = validation_df[LABEL_COLUMN].astype(int)

print(
    "Logistic baseline validation 90-day outcome ROC-AUC:",
    roc_auc_score(validation_y, logistic_validation_probabilities),
)
print(
    "Logistic baseline validation 90-day outcome average precision:",
    average_precision_score(validation_y, logistic_validation_probabilities),
)
print(
    "Logistic baseline validation Brier score:",
    brier_score_loss(validation_y, logistic_validation_probabilities),
)

Logistic baseline validation 90-day outcome ROC-AUC: 0.8627305605786617
Logistic baseline validation 90-day outcome average precision: 0.5303466286953371
Logistic baseline validation Brier score: 0.11834991428419404


## Training

In [20]:
trainer = DecisionTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=lead_train_items,
    eval_dataset=lead_validation_items,
    head_learning_rate=1e-4,
    args=TrainingArguments(
        per_device_train_batch_size=8,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        num_train_epochs=2,
        learning_rate=2e-4,
        lr_scheduler_type="cosine",
        weight_decay=0.01,
        bf16=is_bfloat16_supported(),
        fp16=not is_bfloat16_supported(),
        eval_strategy="epoch",
        logging_steps=10,
        output_dir="outputs/lead-decision",
        report_to="none",
        seed=3407,
    ),
)

In [21]:
_train_start = time.perf_counter()

In [22]:
trainer_stats = trainer.train()
print(trainer_stats.metrics)

Unsloth: Will smartly offload gradients to save VRAM!


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Epoch,Training Loss,Validation Loss
1,0.174437,0.198233
2,0.172389,0.176219


Unsloth: Restored added_tokens_decoder metadata in /home/dev/work/lead-decision-models/outputs/lead-decision/checkpoint-150/.unsloth-save-rs_gwr97/tokenizer_config.json.


{'train_runtime': 917.9405, 'train_samples_per_second': 5.229, 'train_steps_per_second': 0.163, 'total_flos': 0.0, 'train_loss': 0.23514798800150552, 'epoch': 2.0}


In [23]:
TIMINGS["lead_training"] = round(time.perf_counter() - _train_start, 1)
peak_vram_gb = round(torch.cuda.max_memory_allocated() / 1024**3, 2)
print("Training seconds:", TIMINGS["lead_training"], "| peak VRAM GB:", peak_vram_gb)

Training seconds: 969.0 | peak VRAM GB: 2.73


## Calibration and evaluation

In [24]:
lead_calibration_metrics = FastDecisionModel.calibrate(
    model,
    tokenizer,
    lead_calibration_items,
)

print(lead_calibration_metrics)

{'accuracy': 0.8954849498327759, 'ece': 0.00851519022099544, 'loss': 0.15697780038205314, 'record_accuracy': 0.7909698996655519, 'fitted_types': [0, 2]}


In [25]:
lead_validation_metrics = FastDecisionModel.evaluate(
    model,
    tokenizer,
    lead_validation_items,
)

print("Fine-tuned validation metrics:", lead_validation_metrics)

Fine-tuned validation metrics: {'accuracy': 0.895, 'ece': 0.009118761748075473, 'loss': 0.1768190702890835, 'record_accuracy': 0.79}


In [26]:
_predict_start = time.perf_counter()

In [27]:
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    classification_report,
    roc_auc_score,
)

def option_probability(probabilities, option):
    for key, value in probabilities.items():
        if str(key).lower() == option.lower():
            return float(value)
    raise ValueError(f"No {option!r} probability in {probabilities}")

def true_probability(probabilities):
    return option_probability(probabilities, "true")

def predict_leads(frame, decision_model, decision_tokenizer):
    predictions = []

    for row in frame.to_dict(orient="records"):
        example = make_lead_record(row)

        result = FastDecisionModel.predict(
            decision_model,
            decision_tokenizer,
            example["state"],
            LEAD_QUESTIONS,
        )

        predictions.append(
            {
                "icp_fit": result["icp_fit"]["answer"],
                "icp_fit_probability": option_probability(
                    result["icp_fit"]["probabilities"],
                    "fit",
                ),
                "conversion_outcome_prediction": result["conversion_outcome_90d"]["answer"],
                "outcome_probability": true_probability(
                    result["conversion_outcome_90d"]["probabilities"]
                ),
            }
        )

    return pd.DataFrame(predictions)

def precision_at_k(y_true, scores, k=100):
    k = min(k, len(y_true))
    selected = np.argsort(scores)[-k:]
    return float(np.asarray(y_true)[selected].mean())

FastDecisionModel.for_inference(model)

lead_validation_predictions = predict_leads(
    validation_df,
    model,
    tokenizer,
)

validation_icp_labels = validation_df.apply(icp_label, axis=1)

print(
    classification_report(
        validation_icp_labels,
        lead_validation_predictions["icp_fit"],
        labels=["fit", "not_fit", "insufficient_evidence"],
        zero_division=0,
    )
)

validation_y = validation_df[LABEL_COLUMN].astype(int)
validation_scores = lead_validation_predictions["outcome_probability"]

print("90-day outcome ROC-AUC:", roc_auc_score(validation_y, validation_scores))
print("90-day outcome average precision:", average_precision_score(validation_y, validation_scores))
print("90-day outcome Brier score:", brier_score_loss(validation_y, validation_scores))
print("90-day outcome Precision@100:", precision_at_k(validation_y, validation_scores, 100))

                       precision    recall  f1-score   support

                  fit       1.00      1.00      1.00       683
              not_fit       1.00      1.00      1.00       317
insufficient_evidence       0.00      0.00      0.00         0

             accuracy                           1.00      1000
            macro avg       0.67      0.67      0.67      1000
         weighted avg       1.00      1.00      1.00      1000

90-day outcome ROC-AUC: 0.8577305605786618
90-day outcome average precision: 0.504131243347345
90-day outcome Brier score: 0.12017684686999999
90-day outcome Precision@100: 0.49


In [28]:
mean_predict_ms = round((time.perf_counter() - _predict_start) * 1000 / len(validation_df), 1)
print("Mean milliseconds per lead prediction:", mean_predict_ms)

Mean milliseconds per lead prediction: 81.1


In [29]:
lead_test_metrics = FastDecisionModel.evaluate(
    model,
    tokenizer,
    lead_test_items,
)

FastDecisionModel.for_inference(model)

lead_test_predictions = predict_leads(
    test_df,
    model,
    tokenizer,
)

test_icp_labels = test_df.apply(icp_label, axis=1)

print(
    classification_report(
        test_icp_labels,
        lead_test_predictions["icp_fit"],
        labels=["fit", "not_fit", "insufficient_evidence"],
        zero_division=0,
    )
)

test_y = test_df[LABEL_COLUMN].astype(int)
test_scores = lead_test_predictions["outcome_probability"]

print("Unsloth test metrics:", lead_test_metrics)
print("90-day outcome ROC-AUC:", roc_auc_score(test_y, test_scores))
print("90-day outcome average precision:", average_precision_score(test_y, test_scores))
print("90-day outcome Brier score:", brier_score_loss(test_y, test_scores))
print("90-day outcome Precision@100:", precision_at_k(test_y, test_scores, 100))

logistic_test_probabilities = logistic_baseline.predict_proba(
    test_df[MODEL_FEATURES]
)[:, positive_class_index]

print(
    "Logistic baseline test 90-day outcome ROC-AUC:",
    roc_auc_score(test_y, logistic_test_probabilities),
)
print(
    "Logistic baseline test 90-day outcome average precision:",
    average_precision_score(test_y, logistic_test_probabilities),
)
print(
    "Logistic baseline test Brier score:",
    brier_score_loss(test_y, logistic_test_probabilities),
)
print(
    "Logistic baseline test 90-day outcome Precision@100:",
    precision_at_k(test_y, logistic_test_probabilities, 100),
)

                       precision    recall  f1-score   support

                  fit       1.00      1.00      1.00       672
              not_fit       1.00      1.00      1.00       330
insufficient_evidence       0.00      0.00      0.00         0

             accuracy                           1.00      1002
            macro avg       0.67      0.67      0.67      1002
         weighted avg       1.00      1.00      1.00      1002

Unsloth test metrics: {'accuracy': 0.8947105788423154, 'ece': 0.005664610755657736, 'loss': 0.15573090625445243, 'record_accuracy': 0.7894211576846307}
90-day outcome ROC-AUC: 0.8603633291592021
90-day outcome average precision: 0.49516830394822975
90-day outcome Brier score: 0.10920424804391217
90-day outcome Precision@100: 0.45
Logistic baseline test 90-day outcome ROC-AUC: 0.8757526917154481
Logistic baseline test 90-day outcome average precision: 0.5495348201164751
Logistic baseline test Brier score: 0.11052304954493032
Logistic baseline test 90-

## Save, publish and release

In [30]:
model.save_pretrained("outputs/lead-decision-adapter")
model.save_pretrained_merged(
    "outputs/lead-decision-merged",
    tokenizer,
)

Unsloth: Restored added_tokens_decoder metadata in /home/dev/work/lead-decision-models/outputs/lead-decision-adapter/.unsloth-save-h38g5n4h/tokenizer_config.json.


Unsloth: Restored added_tokens_decoder metadata in /home/dev/work/lead-decision-models/outputs/lead-decision-merged/.unsloth-save-cclr14ki/tokenizer_config.json.


/home/dev/work/lead-decision-models/unsloth_env/lib/python3.13/site-packages/unsloth_zoo/saving_utils.py:5436: UserWarning: Unsloth: Could not read the base config from `LiquidAI/LFM3-1.7B` (The repository LiquidAI/LFM3-1.7B contains custom code which must be executed to correctly load the model. You can inspect the repository content at https://hf.co/LiquidAI/LFM3-1.7B .
 You can inspect the repository content at https://hf.co/LiquidAI/LFM3-1.7B.
Please pass the argument `trust_remote_code=True` to allow custom code to be run.). Falling back to the in-memory config, which might not describe the exported weights (see #969).
  warnings.warn(


/home/dev/work/lead-decision-models/unsloth_env/lib/python3.13/site-packages/unsloth_zoo/saving_utils.py:3849: UserWarning: Unsloth: `LiquidAI/LFM3-1.7B` was loaded without a recorded commit, so its repo code was not copied into the export. Copy the repo's *.py files at the revision you loaded into `/home/dev/work/lead-decision-models/outputs/lead-decision-merged/.unsloth-save-cclr14ki` before loading it with trust_remote_code=True.
  warnings.warn(


Found HuggingFace hub cache directory: /home/dev/.cache/huggingface/hub
Checking cache directory for required files...


Unsloth: Copying 1 files from cache to `/home/dev/work/lead-decision-models/outputs/lead-decision-merged/.unsloth-save-cclr14ki`:   0%|          | 0/1 [00:00<?, ?it/s]

... (6 outputs left out)

Unsloth: Merging weights into 16bit:   0%|          | 0/1 [00:00<?, ?it/s]

Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:03<00:00,  3.74s/it]

Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:03<00:00,  3.74s/it]

Unsloth: Merge process complete. Saved to `/home/dev/work/lead-decision-models/outputs/lead-decision-merged/.unsloth-save-cclr14ki`


In [31]:
LEAD_REPO = f"{HF_USERNAME}/lead-decision-models-lead-qualifier"

if PUBLISH:
    model.push_to_hub(LEAD_REPO, tokenizer)
    model.push_to_hub_merged(f"{LEAD_REPO}-merged", tokenizer, save_method="merged_16bit")
    print("Published", LEAD_REPO, "and", f"{LEAD_REPO}-merged")

Unsloth: Restored added_tokens_decoder metadata in /tmp/tmpnf9a56zg/.unsloth-save-m95e3x3a/tokenizer_config.json.


Unsloth: Saved the decision model to https://huggingface.co/raoofaltaher/lead-decision-models-lead-qualifier


Unsloth: Restored added_tokens_decoder metadata in /tmp/tmp0psfp2bx/.unsloth-save-ju9t2nr7/tokenizer_config.json.


/home/dev/work/lead-decision-models/unsloth_env/lib/python3.13/site-packages/unsloth_zoo/saving_utils.py:3849: UserWarning: Unsloth: `LiquidAI/LFM3-1.7B` was loaded without a recorded commit, so its repo code was not copied into the export. Copy the repo's *.py files at the revision you loaded into `/tmp/tmp0psfp2bx/.unsloth-save-ju9t2nr7` before loading it with trust_remote_code=True.
  warnings.warn(


Found HuggingFace hub cache directory: /home/dev/.cache/huggingface/hub
Checking cache directory for required files...


Unsloth: Copying 1 files from cache to `/tmp/tmp0psfp2bx/.unsloth-save-ju9t2nr7`:   0%|          | 0/1 [00:00<?, ?it/s]

... (6 outputs left out)

Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:02<00:00,  2.94s/it]

Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:02<00:00,  2.94s/it]

Unsloth: Merge process complete. Saved to `/tmp/tmp0psfp2bx/.unsloth-save-ju9t2nr7`


Unsloth: Saved the decision model to https://huggingface.co/raoofaltaher/lead-decision-models-lead-qualifier-merged
Published raoofaltaher/lead-decision-models-lead-qualifier and raoofaltaher/lead-decision-models-lead-qualifier-merged


In [32]:
import gc

del trainer, model, tokenizer, trainer_stats
gc.collect()
torch.cuda.empty_cache()

## Outreach policy

In [33]:
TARGET_BUYER_ROLES = {
    "economic_buyer",
    "champion",
    "technical_evaluator",
}

def evaluate_contact_persona_fit(buyer_role):
    if pd.isna(buyer_role):
        return None

    return str(buyer_role).strip().lower() in TARGET_BUYER_ROLES

In [34]:
def lead_action(
    prediction,
    suppressed,
    contact_persona_fit,
    contact_eligibility,
):
    if isinstance(suppressed, np.bool_):
        suppressed = bool(suppressed)
    if isinstance(contact_persona_fit, np.bool_):
        contact_persona_fit = bool(contact_persona_fit)

    if suppressed is True:
        return {
            "priority": "blocked",
            "action": "suppress",
            "send": False,
        }

    if suppressed is not False:
        return {
            "priority": "review",
            "action": "verify_suppression_status",
            "send": False,
        }

    fit = prediction.get("icp_fit")

    if fit not in {"fit", "not_fit", "insufficient_evidence"}:
        return {
            "priority": "review",
            "action": "human_icp_review",
            "send": False,
        }

    if fit == "insufficient_evidence":
        return {
            "priority": "review",
            "action": "research_missing_account_evidence",
            "send": False,
        }

    if fit == "not_fit":
        return {
            "priority": "low",
            "action": "do_not_recommend_outreach",
            "send": False,
        }

    if contact_persona_fit is False:
        return {
            "priority": "high",
            "icp_fit_probability": prediction.get("icp_fit_probability"),
            "action": "find_relevant_contact",
            "send": False,
        }

    if contact_persona_fit is not True:
        return {
            "priority": "high",
            "icp_fit_probability": prediction.get("icp_fit_probability"),
            "action": "human_persona_review",
            "send": False,
        }

    if contact_eligibility == "ineligible":
        return {
            "priority": "blocked",
            "action": "do_not_contact",
            "send": False,
        }

    if contact_eligibility != "approved":
        return {
            "priority": "high",
            "icp_fit_probability": prediction.get("icp_fit_probability"),
            "action": "human_eligibility_review",
            "send": False,
        }

    return {
        "priority": "high",
        "icp_fit_probability": prediction.get("icp_fit_probability"),
        "action": "recommend_outreach_for_approval",
        "send": False,
    }

In [35]:
sample_lead = lead_validation_predictions.iloc[0].to_dict()

print(
    lead_action(
        sample_lead,
        suppressed=False,
        contact_persona_fit=evaluate_contact_persona_fit("champion"),
        contact_eligibility="unknown",
    )
)

{'priority': 'low', 'action': 'do_not_recommend_outreach', 'send': False}


In [36]:
fit_prediction = {"icp_fit": "fit"}

assert lead_action(
    fit_prediction,
    suppressed=True,
    contact_persona_fit=True,
    contact_eligibility="approved",
)["action"] == "suppress"

assert lead_action(
    fit_prediction,
    suppressed=None,
    contact_persona_fit=True,
    contact_eligibility="approved",
)["action"] == "verify_suppression_status"

assert lead_action(
    fit_prediction,
    suppressed=False,
    contact_persona_fit=True,
    contact_eligibility="ineligible",
)["action"] == "do_not_contact"

assert lead_action(
    {"icp_fit": "unexpected_label"},
    suppressed=False,
    contact_persona_fit=True,
    contact_eligibility="approved",
)["action"] == "human_icp_review"

assert lead_action(
    {"icp_fit": "fit"},
    suppressed=np.bool_(True),
    contact_persona_fit=True,
    contact_eligibility="approved",
)["action"] == "suppress"

## Reload the merged model

In [37]:
reloaded_lead_model, reloaded_lead_tokenizer = FastDecisionModel.from_pretrained(
    model_name="outputs/lead-decision-merged",
    trust_remote_code=True,
    max_seq_length=2048,
    load_in_4bit=LOAD_IN_4BIT,
)

FastDecisionModel.for_inference(reloaded_lead_model)

lead_example = make_lead_record(test_df.iloc[0])
lead_reload_prediction = FastDecisionModel.predict(
    reloaded_lead_model,
    reloaded_lead_tokenizer,
    lead_example["state"],
    LEAD_QUESTIONS,
)

print(lead_reload_prediction)

lead_before_export = lead_test_predictions.iloc[0]
print(
    "Lead class agreement after export:",
    lead_reload_prediction["icp_fit"]["answer"] == lead_before_export["icp_fit"],
    lead_reload_prediction["conversion_outcome_90d"]["answer"]
    == bool(lead_before_export["conversion_outcome_prediction"]),
)

Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2026.10.11: Fast Lfm3 patching. Transformers: 5.22.1.
   \\   /|    NVIDIA GeForce RTX 5090 Laptop GPU. Num GPUs = 1. Max memory: 23.863 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.14.1+cu132. CUDA: 12.0. CUDA Toolkit: 13.2. Triton: 3.8.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/286 [00:00<?, ?it/s]

Unsloth: Repaired remote modeling code so it trains: Lfm3ForCausalLM.forward.


Unsloth: compiling each transformer block as one region (Lfm3DecoderLayer).


{'icp_fit': {'type': 'choice', 'choice': 'fit', 'confidence': 1.0, 'probabilities': {'fit': 1.0, 'not_fit': 0.0, 'insufficient_evidence': 0.0}, 'answer': 'fit'}, 'conversion_outcome_90d': {'type': 'noul', 'noul': 0.003, 'answer': False, 'probabilities': {'true': 0.003, 'false': 0.997}}}
Lead class agreement after export: True True


In [38]:
del reloaded_lead_model, reloaded_lead_tokenizer
gc.collect()
torch.cuda.empty_cache()

## Metrics file

In [39]:
import json

def outcome_metrics(y_true, scores):
    return {
        "roc_auc": float(roc_auc_score(y_true, scores)),
        "average_precision": float(average_precision_score(y_true, scores)),
        "brier": float(brier_score_loss(y_true, scores)),
        "precision_at_100": float(precision_at_k(y_true, scores, 100)),
    }

lead_metrics = {
    "model_id": MODEL_ID,
    "model_revision": MODEL_REVISION,
    "load_in_4bit": LOAD_IN_4BIT,
    "dataset": {
        "leadforge_commit": DATASET_COMMIT,
        "target_counts": {str(key): int(value) for key, value in leads[LABEL_COLUMN].value_counts().items()},
    },
    "splits": {
        "optimizer_training": len(lead_fit_df),
        "calibration": len(lead_calibration_df),
        "validation": len(validation_df),
        "test": len(test_df),
    },
    "baseline_untrained": lead_baseline,
    "logistic": {
        "validation": outcome_metrics(validation_y, logistic_validation_probabilities),
        "test": outcome_metrics(test_y, logistic_test_probabilities),
    },
    "calibration": lead_calibration_metrics,
    "validation": {**lead_validation_metrics, **outcome_metrics(validation_y, validation_scores)},
    "test": {**lead_test_metrics, **outcome_metrics(test_y, test_scores)},
    "icp_report": {
        "validation": classification_report(validation_icp_labels, lead_validation_predictions["icp_fit"], output_dict=True, zero_division=0),
        "test": classification_report(test_icp_labels, lead_test_predictions["icp_fit"], output_dict=True, zero_division=0),
    },
    "timings_seconds": TIMINGS,
    "peak_vram_gb": peak_vram_gb,
    "mean_predict_ms": mean_predict_ms,
}

Path("outputs").mkdir(exist_ok=True)
Path("outputs/lead-metrics.json").write_text(json.dumps(lead_metrics, indent=2, default=float), encoding="utf-8")
print(json.dumps({key: lead_metrics[key] for key in ["splits", "logistic", "test", "peak_vram_gb", "mean_predict_ms", "timings_seconds"]}, indent=2, default=float))

{
  "splits": {
    "optimizer_training": 2400,
    "calibration": 598,
    "validation": 1000,
    "test": 1002
  },
  "logistic": {
    "validation": {
      "roc_auc": 0.8627305605786617,
      "average_precision": 0.5303466286953371,
      "brier": 0.11834991428419404,
      "precision_at_100": 0.61
    },
    "test": {
      "roc_auc": 0.8757526917154481,
      "average_precision": 0.5495348201164751,
      "brier": 0.11052304954493032,
      "precision_at_100": 0.54
    }
  },
  "test": {
    "accuracy": 0.8947105788423154,
    "ece": 0.005664610755657736,
    "loss": 0.15573090625445243,
    "record_accuracy": 0.7894211576846307,
    "roc_auc": 0.8603633291592021,
    "average_precision": 0.49516830394822975,
    "brier": 0.10920424804391217,
    "precision_at_100": 0.45
  },
  "peak_vram_gb": 2.73,
  "mean_predict_ms": 81.1,
  "timings_seconds": {
    "lead_training": 969.0
  }
}
